# Notebook 03 — Model A: Nowcast Level Kemacetan dari Video
**Proyek:** Sistem Pemantauan & Prediksi Kemacetan Lalu Lintas Kota berbasis IoT + Big Data (Lambda Architecture)

## Posisi notebook ini (Speed Layer)
```
 Video CCTV ─► YOLOv8 ─► ByteTrack ─► FITUR LALU LINTAS ─► MODEL A ─► "Lancar / Padat / Macet" (real-time)
              deteksi     ID per        kecepatan, kepadatan,   klasifier        ↓
                          kendaraan     okupansi, arus                       dashboard
```
Notebook 02 mengajari model **melihat** kendaraan. Notebook ini mengubah apa yang dilihat menjadi **angka lalu lintas**, lalu memutuskan **seberapa macet** kondisinya.

## Yang akan kamu kuasai
1. Menyadari dan menangani **domain shift** — model yang bagus di satu kota bisa buta di kota lain.
2. **Feature engineering** dari tracking: mengubah kotak bergerak menjadi kecepatan, kepadatan, okupansi.
3. Membuat fitur yang **tidak bergantung kalibrasi kamera** (penting supaya model bisa dipakai di kamera lain).
4. Evaluasi jujur dengan **split resmi dataset** dan **macro-F1**.
5. Menjalankan speed layer di video uji: **sliding window** → timeline level kemacetan.

> ⚠️ **Gunakan GPU:** Runtime → Ubah jenis runtime → T4 GPU.

## 0. Setup

In [ ]:
import sys, os, shutil, glob, hashlib, json, time, zipfile, random
IN_COLAB = "google.colab" in sys.modules
%pip install -q ultralytics lap scikit-learn     # pesan "restart the kernel" boleh diabaikan

import torch
GPU = torch.cuda.is_available(); DEVICE = 0 if GPU else "cpu"
print("Colab:", IN_COLAB, "| GPU:", torch.cuda.get_device_name(0) if GPU else "TIDAK ADA")
TEST_MODE = os.environ.get("NB_TEST") == "1"          # hanya untuk uji notebook tanpa GPU

### Lokasi file
Upload ke Drive (lewat drive.google.com, bukan panel File Colab):
```
MyDrive/Projek Akhir/
├── Dataset/
│   ├── ucsd_traffic.zip         ← 254 video UCSD + label (32 MB)
│   └── video_uji.mp4            ← sudah ada dari notebook 02
└── output/02_yolo/
    └── yolov8s_traffic7.pt      ← sudah ada dari notebook 02
```

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/Projek Akhir"     # <-- sesuaikan bila perlu
    WORK = "/content"
else:
    BASE = os.environ.get("PROJ_BASE", ".")
    WORK = os.environ.get("PROJ_WORK", "./work")

DATA_DIR  = f"{BASE}/Dataset"
OUT_DIR   = f"{BASE}/output/03_model_a"
UCSD_ZIP  = f"{DATA_DIR}/ucsd_traffic.zip"
UCSD_MD5  = os.environ.get("NB_MD5", "d911e791591897135e3441c3370c390b")
FT_MODEL  = f"{BASE}/output/02_yolo/yolov8s_traffic7.pt"
VIDEO     = f"{DATA_DIR}/video_uji.mp4"
UCSD_DIR  = f"{WORK}/ucsd"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
for p in [UCSD_ZIP, VIDEO]:
    print(("✅ " if os.path.exists(p) else "❌ TIDAK ADA: ") + p)
print(("✅ " if os.path.exists(FT_MODEL) else "⚠️  (opsional) tidak ada: ") + FT_MODEL)
assert os.path.exists(UCSD_ZIP), "Upload ucsd_traffic.zip dulu"

def savefig(name): plt.savefig(f"{OUT_DIR}/figures/{name}.png", bbox_inches="tight")

In [ ]:
h = hashlib.md5(open(UCSD_ZIP, "rb").read()).hexdigest()
print("MD5 file  :", h); print("MD5 laptop:", UCSD_MD5 or "(tidak diisi)")
if UCSD_MD5: assert h == UCSD_MD5, "❌ File rusak saat upload — upload ulang ucsd_traffic.zip"
if os.path.exists(UCSD_DIR): shutil.rmtree(UCSD_DIR)
zipfile.ZipFile(UCSD_ZIP).extractall(WORK)
print("✅ Diekstrak:", len(os.listdir(f"{UCSD_DIR}/video")), "video")

## 1. Memahami dataset UCSD Traffic
- **254 klip** dari satu kamera tetap WSDOT di jalan tol I-5, Seattle (Agustus 2004), **320×240 piksel, 10 fps, ±5 detik**.
- **Label manual** per klip: `light` (arus bebas), `medium` (kecepatan berkurang), `heavy` (berhenti / sangat lambat).
- Disediakan **4 pasang split train/test resmi** — kita wajib memakai ini supaya hasil bisa dibandingkan dengan paper.
- Catatan dari README: **frame pertama setiap video rusak**, jadi kita mulai dari frame ke-2.

Kita tampilkan label ke pengguna dashboard dengan istilah yang mudah dipahami city planner:

| Label UCSD | Tampilan dashboard | Padanan level PEMS (notebook 01) |
|---|---|---|
| light | **Lancar** | Low |
| medium | **Padat** | Moderate – High |
| heavy | **Macet** | Severe |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
from collections import Counter

LABELS  = ["light", "medium", "heavy"]
DISPLAY = {"light": "Lancar", "medium": "Padat", "heavy": "Macet"}
COLORS  = {"light": "#54A24B", "medium": "#F58518", "heavy": "#E45756"}

cols = ["file", "date", "timestamp", "direction", "daynight", "weather", "start_frame", "n_frames", "label", "notes"]
info = pd.read_csv(f"{UCSD_DIR}/info.txt", sep="\t", comment="#", names=cols)
im   = pd.read_csv(f"{UCSD_DIR}/ImageMaster", sep="\t", comment="#", names=["idx", "file", "label"])
read_split = lambda p: [list(map(int, l.split(","))) for l in open(p).read().strip().splitlines()]
SPLITS = list(zip(read_split(f"{UCSD_DIR}/EvalSet_train"), read_split(f"{UCSD_DIR}/EvalSet_test")))

print("Jumlah klip per label:", im.label.value_counts().reindex(LABELS).to_dict())
print("Cuaca:", info.weather.value_counts().to_dict())
for k, (tr, te) in enumerate(SPLITS):
    print(f"Split {k + 1}: train {len(tr)} | test {len(te)} | overlap {len(set(tr) & set(te))}")

In [ ]:
def read_clip(path, start=1):
    cap = cv2.VideoCapture(path); frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(f)
    cap.release()
    return frames[start:]            # buang frame pertama yang rusak

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, lab in zip(axes, LABELS):
    f = im[im.label == lab].file.iloc[3]
    ax.imshow(read_clip(f"{UCSD_DIR}/video/{f}.avi")[10][..., ::-1]); ax.axis("off")
    ax.set_title(f"{lab} → {DISPLAY[lab]}", color=COLORS[lab], fontweight="bold")
savefig("01_contoh_klip"); plt.show()

## 2. Domain shift: detektor mana yang dipakai?
Model hasil fine-tuning di notebook 02 dilatih dari CCTV kota di Asia Selatan (malam hari, banyak becak/bajaj/motor). Jalan tol Seattle tahun 2004 sangat berbeda: siang, resolusi rendah, hanya mobil & truk.

**Hipotesis:** model fine-tuned bisa *lebih buruk* daripada YOLOv8 bawaan (COCO) di domain ini, karena saat fine-tuning ia "melupakan" sebagian pengetahuan umumnya (*catastrophic forgetting*). Kita uji, bukan menebak.

In [ ]:
from ultralytics import YOLO

coco = YOLO("yolov8s.pt")
COCO_VEH = [2, 3, 5, 7]                       # car, motorcycle, bus, truck
detectors = {"COCO (bawaan)": (coco, COCO_VEH)}
if os.path.exists(FT_MODEL):
    ft = YOLO(FT_MODEL)
    detectors["Fine-tuned (nb 02)"] = (ft, [i for i, n in ft.names.items() if n != "person"])

random.seed(0)
sample = [read_clip(f"{UCSD_DIR}/video/{f}.avi")[20] for lab in LABELS for f in random.sample(list(im[im.label == lab].file), 3)]
counts = {name: [len(m.predict(fr, imgsz=640, conf=0.25, classes=c, verbose=False, device=DEVICE)[0].boxes) for fr in sample]
          for name, (m, c) in detectors.items()}
print("Rata-rata kendaraan terdeteksi per frame (9 frame sampel):")
for name, c in counts.items(): print(f"  {name:20s}: {np.mean(c):5.1f}   {c}")

j = int(np.argsort(counts["COCO (bawaan)"])[len(sample) // 2])        # frame representatif (median), bukan dipilih-pilih
fig, axes = plt.subplots(1, len(detectors), figsize=(7 * len(detectors), 4))
for ax, (name, (m, c)) in zip(np.atleast_1d(axes), detectors.items()):
    r = m.predict(sample[j], imgsz=640, conf=0.25, classes=c, verbose=False, device=DEVICE)[0]
    ax.imshow(r.plot(line_width=1, font_size=6)[..., ::-1]); ax.set_title(f"{name}: {len(r.boxes)} kendaraan"); ax.axis("off")
savefig("02_domain_shift"); plt.show()

**Kesimpulan (tulis di laporan):** di jalan tol Barat, YOLOv8 bawaan jauh lebih baik. Model fine-tuned tetap berguna untuk video lalu lintas campuran ala Indonesia (motor, bajaj, becak). Jadi sistem kita **memilih detektor sesuai domain kamera** — praktik yang memang dipakai di sistem nyata (satu model per tipe lokasi kamera).

Untuk melatih Model A di UCSD, kita pakai **COCO**.

## 3. Feature engineering: dari kotak bergerak menjadi angka lalu lintas
Masalah utama: kita **tidak tahu ukuran asli** jalan di video (tidak ada kalibrasi meter/piksel). Kecepatan dalam *piksel/detik* tidak bisa dibandingkan antar kamera — mobil yang jauh terlihat lambat, mobil dekat terlihat cepat.

**Trik:** ukur kecepatan dalam **"panjang kendaraan per detik"** (*body-lengths per second*, bl/s). Perpindahan kotak dibagi tinggi kotak itu sendiri. Mobil jauh: kotaknya kecil **dan** gerak pikselnya kecil → rasionya tetap sebanding. Hasilnya hampir tidak bergantung pada zoom/resolusi kamera.

| Fitur | Arti lalu lintas | Tidak bergantung kamera? |
|---|---|---|
| `speed_bl_med` | median kecepatan kendaraan (bl/s) | ✅ |
| `speed_bl_p25` | kecepatan kuartil bawah — menangkap antrean | ✅ |
| `stopped_ratio` | % kendaraan hampir berhenti (< 0,5 bl/s) | ✅ |
| `optflow_bl` | kecepatan dari *optical flow* piksel (cadangan saat tracking gagal) | ✅ |
| `occupancy` | % area jalan tertutup kendaraan (≈ kepadatan) | ✅ (kira-kira) |
| `track_len_ratio` | berapa lama kendaraan tetap terlihat — lambat = lama | ✅ |
| `det_mean`, `tracks_per_s`, `flow_per_min` | jumlah kendaraan, arus | ❌ tergantung luas pandang kamera |

Fitur ❌ tetap kita hitung untuk analisis dan dashboard, tapi **tidak dipakai Model A**, supaya model bisa dipindah ke kamera lain.

In [ ]:
def reset_tracker(model):
    p = getattr(model, "predictor", None)
    if p is not None and getattr(p, "trackers", None):
        for t in p.trackers: t.reset()                 # tracker baru untuk setiap klip/jendela

def extract_features(frames, model, classes, fps=10, roi_top=0.35, line_y=0.70, imgsz=640, conf=0.2, keep_plots=False):
    '''frames: list gambar BGR berurutan → dict fitur lalu lintas'''
    H, W = frames[0].shape[:2]; y0 = int(roi_top * H); ly = line_y * H; roi_area = (H - y0) * W
    reset_tracker(model)
    tracks, dets, occ, flows, plots, prev = {}, [], [], [], [], None
    for t, f in enumerate(frames):
        r = model.track(f, persist=True, tracker="bytetrack.yaml", imgsz=imgsz, conf=conf,
                        classes=classes, verbose=False, device=DEVICE)[0]
        mask = np.zeros((H, W), np.uint8); n = 0
        if r.boxes is not None and len(r.boxes):
            xyxy = r.boxes.xyxy.cpu().numpy()
            ids = r.boxes.id.int().cpu().numpy() if r.boxes.id is not None else [None] * len(xyxy)
            for (x1, y1, x2, y2), tid in zip(xyxy, ids):
                cx, cy, h = (x1 + x2) / 2, (y1 + y2) / 2, y2 - y1
                if cy < y0: continue                          # abaikan area jauh (kotak terlalu kecil)
                n += 1; mask[int(max(y1, y0)):int(y2), int(x1):int(x2)] = 1
                if tid is not None: tracks.setdefault(int(tid), []).append((t, cx, cy, h))
        dets.append(n); occ.append(mask[y0:].sum() / roi_area)
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        if prev is not None:
            fl = cv2.calcOpticalFlowFarneback(prev[y0:], gray[y0:], None, 0.5, 3, 15, 3, 5, 1.2, 0)
            mag = np.linalg.norm(fl, axis=2); veh = mask[y0:] > 0
            flows.append(np.median(mag[veh]) if veh.sum() > 50 else np.nan)
        prev = gray
        if keep_plots: plots.append(r.plot(line_width=1, font_size=6))

    dur = len(frames) / fps; speeds, crossings, lens = [], 0, []
    for pts in tracks.values():
        pts = np.array(pts); lens.append(len(pts))
        if len(pts) >= 5:
            dt = (pts[-1, 0] - pts[0, 0]) / fps
            disp = np.hypot(pts[-1, 1] - pts[0, 1], pts[-1, 2] - pts[0, 2])
            speeds.append(disp / np.median(pts[:, 3]) / dt)          # panjang-kendaraan per detik
        if pts[:, 2].min() < ly <= pts[:, 2].max(): crossings += 1   # melewati garis hitung virtual
    speeds = np.array(speeds) if speeds else np.array([np.nan])
    med_h = np.nanmedian([p[3] for v in tracks.values() for p in v]) if tracks else np.nan
    ok = np.isfinite(speeds).any()
    feats = {
        "speed_bl_med":    float(np.nanmedian(speeds)) if ok else np.nan,
        "speed_bl_p25":    float(np.nanpercentile(speeds, 25)) if ok else np.nan,
        "stopped_ratio":   float(np.mean(speeds[np.isfinite(speeds)] < 0.5)) if ok else np.nan,
        "optflow_bl":      float(np.nanmedian(flows) / med_h * fps) if flows and np.isfinite(med_h) and np.isfinite(flows).any() else np.nan,
        "occupancy":       float(np.mean(occ)),
        "track_len_ratio": float(np.mean(lens) / len(frames)) if lens else 0.0,
        "det_mean":        float(np.mean(dets)),
        "tracks_per_s":    len(tracks) / dur,
        "flow_per_min":    crossings / dur * 60,
    }
    return (feats, plots) if keep_plots else feats

f0 = im.file.iloc[0]
print(f0, im.label.iloc[0], {k: round(v, 2) for k, v in extract_features(read_clip(f"{UCSD_DIR}/video/{f0}.avi"), coco, COCO_VEH).items()})

### 3.1 Ekstraksi fitur semua klip
Hasil disimpan ke Drive setiap 25 klip → kalau Colab terputus, jalankan ulang dan proses **melanjutkan**, tidak mengulang dari awal. Di GPU T4 perkiraan ±5–10 menit.

In [ ]:
FEAT_CSV = f"{OUT_DIR}/ucsd_features.csv"
clips = im if not TEST_MODE else pd.concat([d.head(10) for _, d in im.groupby("label")])
rows = pd.read_csv(FEAT_CSV).to_dict("records") if os.path.exists(FEAT_CSV) else []
done = {r["file"] for r in rows}
t0 = time.time()
for _, r in clips.iterrows():
    if r.file in done: continue
    feats = extract_features(read_clip(f"{UCSD_DIR}/video/{r.file}.avi"), coco, COCO_VEH)
    rows.append({"idx": r.idx, "file": r.file, "label": r.label, **feats})
    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(FEAT_CSV, index=False)
        print(f"{len(rows):4d}/{len(clips)} klip | {time.time() - t0:5.0f} s")
feats = pd.DataFrame(rows); feats.to_csv(FEAT_CSV, index=False)
print(f"Selesai: {len(feats)} klip | {time.time() - t0:.0f} s")
feats.groupby("label")[["speed_bl_med", "stopped_ratio", "occupancy", "det_mean", "flow_per_min"]].median().reindex(LABELS).round(2)

### 3.2 Apakah fitur membedakan kelas?
Sebelum melatih model, **lihat datanya**. Fitur yang baik menunjukkan kotak (box) yang terpisah antar kelas.

In [ ]:
show_cols = ["speed_bl_med", "speed_bl_p25", "stopped_ratio", "optflow_bl", "occupancy",
             "track_len_ratio", "det_mean", "tracks_per_s", "flow_per_min"]
fig, axes = plt.subplots(2, 5, figsize=(17, 6)); axes = axes.ravel()
for ax, c in zip(axes, show_cols):
    data = [feats.loc[feats.label == l, c].dropna() for l in LABELS]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6)
    for patch, l in zip(bp["boxes"], LABELS): patch.set_facecolor(COLORS[l])
    ax.set_xticks([1, 2, 3], [DISPLAY[l] for l in LABELS]); ax.set_title(c)
axes[-1].axis("off"); plt.tight_layout(); savefig("03_fitur_per_kelas"); plt.show()

**Bacaan:** fitur kecepatan (`speed_bl_*`, `optflow_bl`) dan `stopped_ratio` paling jelas memisahkan **Macet** dari kelas lain. Memisahkan **Padat** dari **Lancar** lebih sulit — ini wajar, bahkan manusia yang melabel pun kadang ragu di batas ini.

## 4. Melatih Model A
Kita bandingkan beberapa model **dan baseline**. Baseline penting: kalau model rumit tidak mengalahkan aturan sederhana, model rumit itu tidak layak.

- **Baseline mayoritas** — selalu menebak "Lancar" (kelas terbanyak, 65%).
- **Baseline 1 fitur** — hanya `speed_bl_med`.
- **Logistic Regression**, **Random Forest**, **Gradient Boosting** dengan semua fitur ✅.

Kelas tidak seimbang (165 / 45 / 44), jadi semua model memakai `class_weight="balanced"` dan kita menilai dengan **macro-F1** (rata-rata F1 tiap kelas — kelas kecil dihitung sama penting).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import joblib

FEATURES = "speed_bl_med,speed_bl_p25,stopped_ratio,occupancy,track_len_ratio,optflow_bl".split(",")
data = feats.set_index("idx")

def make_models():
    imp = lambda: SimpleImputer(strategy="median")
    return {
        "Baseline mayoritas": (DummyClassifier(strategy="most_frequent"), FEATURES),
        "Baseline 1 fitur (kecepatan)": (make_pipeline(imp(), StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)), ["speed_bl_med"]),
        "Logistic Regression": (make_pipeline(imp(), StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000, C=1.0)), FEATURES),
        "Random Forest": (make_pipeline(imp(), RandomForestClassifier(n_estimators=400, min_samples_leaf=2, class_weight="balanced", random_state=42)), FEATURES),
        "Gradient Boosting": (make_pipeline(imp(), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, class_weight="balanced", random_state=42)), FEATURES),
    }

results, all_preds = [], {}
for name, (model, cols_) in make_models().items():
    y_true_all, y_pred_all = [], []
    for k, (tr, te) in enumerate(SPLITS):
        tr = [i for i in tr if i in data.index]; te = [i for i in te if i in data.index]
        model.fit(data.loc[tr, cols_], data.loc[tr, "label"])
        pred = model.predict(data.loc[te, cols_])
        results.append({"model": name, "split": k + 1,
                        "accuracy": accuracy_score(data.loc[te, "label"], pred),
                        "macro_f1": f1_score(data.loc[te, "label"], pred, average="macro", labels=LABELS, zero_division=0)})
        y_true_all += list(data.loc[te, "label"]); y_pred_all += list(pred)
    all_preds[name] = (y_true_all, y_pred_all)

res = pd.DataFrame(results)
summary = res.groupby("model")[["accuracy", "macro_f1"]].agg(["mean", "std"]).round(3).sort_values(("macro_f1", "mean"), ascending=False)
summary

In [ ]:
candidates = summary.index[~summary.index.str.startswith("Baseline")]
BEST = candidates[0]
print("Model terbaik (macro-F1):", BEST)
y_t, y_p = all_preds[BEST]
print(classification_report(y_t, y_p, labels=LABELS, target_names=[DISPLAY[l] for l in LABELS], digits=3))

cm = confusion_matrix(y_t, y_p, labels=LABELS)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues"); ax.grid(False)
ax.set_xticks(range(3), [DISPLAY[l] for l in LABELS]); ax.set_yticks(range(3), [DISPLAY[l] for l in LABELS])
ax.set_xlabel("Prediksi"); ax.set_ylabel("Label asli"); ax.set_title(f"{BEST}\n(gabungan 4 split test)")
for i in range(3):
    for j in range(3): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
savefig("04_confusion_matrix"); plt.show()

**Cara membaca:** diagonal = benar. Kesalahan yang **paling berbahaya** bagi city planner adalah *Macet* diprediksi *Lancar* (pojok kiri bawah) — kemacetan nyata tidak terlihat di dashboard. Kesalahan *Padat ↔ Lancar* lebih bisa ditoleransi.

### 4.1 Fitur apa yang paling berpengaruh?
*Permutation importance*: acak satu kolom fitur, lihat seberapa jatuh performanya. Makin jatuh = makin penting.

In [ ]:
from sklearn.inspection import permutation_importance
best_model, best_cols = make_models()[BEST]
best_model.fit(data[best_cols], data["label"])
pi = permutation_importance(best_model, data[best_cols], data["label"], scoring="f1_macro", n_repeats=20, random_state=0)
imp = pd.Series(pi.importances_mean, index=best_cols).sort_values()
ax = imp.plot.barh(figsize=(6, 3), color="#4C78A8"); ax.set_xlabel("Penurunan macro-F1 saat fitur diacak")
ax.set_title("Pentingnya fitur — " + BEST); savefig("05_feature_importance"); plt.show()

## 5. Speed layer di video uji: sliding window
Di dunia nyata, video mengalir terus. Speed layer bekerja per **jendela waktu** (*window*): setiap 2,5 detik, ambil 5 detik terakhir → hitung fitur → Model A → kirim level ke dashboard.

Supaya cocok dengan data latih, video diturunkan ke **10 fps** dan lebar 640 piksel (sama seperti klip UCSD yang ±5 detik @10 fps).

**Pemilihan detektor otomatis:** kedua detektor dicoba pada beberapa frame video, lalu dipilih yang mendeteksi lebih banyak kendaraan. Kamu bisa memaksa pilihan dengan mengisi `FORCE_DETECTOR`.

In [ ]:
WINDOW_S, STEP_S, TARGET_FPS = 5.0, 2.5, 10
MAX_SECONDS = 10 if TEST_MODE else 60
FORCE_DETECTOR = None            # None = otomatis | "COCO (bawaan)" | "Fine-tuned (nb 02)"

cap = cv2.VideoCapture(VIDEO); src_fps = cap.get(cv2.CAP_PROP_FPS) or 30
stride = max(1, round(src_fps / TARGET_FPS)); frames, i = [], 0
while True:
    ok, f = cap.read()
    if not ok or len(frames) >= MAX_SECONDS * TARGET_FPS: break
    if i % stride == 0:
        s = 640 / f.shape[1]; frames.append(cv2.resize(f, None, fx=s, fy=s))
    i += 1
cap.release()
fps_eff = src_fps / stride
print(f"{len(frames)} frame @ {fps_eff:.1f} fps ({len(frames) / fps_eff:.1f} detik), ukuran {frames[0].shape[1]}×{frames[0].shape[0]}")

probe = frames[:: max(1, len(frames) // 6)][:6]
det_score = {name: np.mean([len(m.predict(fr, imgsz=640, conf=0.25, classes=c, verbose=False, device=DEVICE)[0].boxes) for fr in probe])
             for name, (m, c) in detectors.items()}
VIDEO_DET = FORCE_DETECTOR or max(det_score, key=det_score.get)
print("Rata-rata deteksi per frame:", {k: round(v, 1) for k, v in det_score.items()}, "→ dipakai:", VIDEO_DET)

In [ ]:
final_model, final_cols = make_models()[BEST]
final_model.fit(data[final_cols], data["label"])
vm, vc = detectors[VIDEO_DET]
win, step = int(WINDOW_S * fps_eff), int(STEP_S * fps_eff)
timeline, segments = [], []
for s0 in range(0, max(1, len(frames) - win + 1), step):
    wf = frames[s0:s0 + win]
    fe, plots = extract_features(wf, vm, vc, fps=fps_eff, keep_plots=True)
    proba = final_model.predict_proba(pd.DataFrame([fe])[final_cols])[0]
    lab = final_model.classes_[proba.argmax()]
    timeline.append({"t_start": s0 / fps_eff, "t_end": (s0 + win) / fps_eff, "level": lab, "display": DISPLAY[lab],
                     **{f"p_{c}": round(p, 3) for c, p in zip(final_model.classes_, proba)}, **{k: round(v, 3) for k, v in fe.items()}})
    segments.append((plots if not segments else plots[-step:], lab, proba.max()))
tl = pd.DataFrame(timeline)
tl[["t_start", "t_end", "display", "p_light", "p_medium", "p_heavy", "speed_bl_med", "stopped_ratio", "occupancy", "det_mean"]]

In [ ]:
out_vid = f"{OUT_DIR}/demo_model_a.mp4"
h0, w0 = segments[0][0][0].shape[:2]
writer = cv2.VideoWriter(out_vid, cv2.VideoWriter_fourcc(*"mp4v"), fps_eff, (w0, h0))
hex2bgr = lambda h: tuple(int(h[i:i + 2], 16) for i in (5, 3, 1))
for plots, lab, conf_ in segments:
    for fr in plots:
        fr = fr.copy(); cv2.rectangle(fr, (0, 0), (w0, 34), hex2bgr(COLORS[lab]), -1)
        cv2.putText(fr, f"Model A: {DISPLAY[lab].upper()}  ({conf_:.0%})", (10, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        writer.write(fr)
writer.release()
tl.to_csv(f"{OUT_DIR}/timeline_video_uji.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 1.6))
for _, r in tl.iterrows():
    ax.barh(0, r.t_end - r.t_start, left=r.t_start, color=COLORS[r.level], alpha=0.6, edgecolor="white")
ax.set_yticks([]); ax.set_xlabel("detik"); ax.set_title("Timeline level kemacetan — video uji")
handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[l]) for l in LABELS]
ax.legend(handles, [DISPLAY[l] for l in LABELS], loc="upper center", bbox_to_anchor=(0.5, -0.45), ncol=3, frameon=False)
savefig("06_timeline_video_uji"); plt.show()
print("Video beranotasi:", out_vid)

> ⚠️ **Batasan yang jujur:** video uji tidak punya label, jadi timeline ini adalah **demonstrasi**, bukan bukti akurasi. Cek manual sederhana: bandingkan dua frame berjarak 2 detik — kalau kendaraan hanya maju ±1 panjang mobil, antrean memang sangat lambat. Random Forest juga cenderung **terlalu yakin** (probabilitas mendekati 100%); jangan tafsirkan angka itu sebagai kepastian. Untuk evaluasi di lalu lintas Indonesia, labeli sendiri beberapa klip lokal (per 5 detik: Lancar/Padat/Macet) lalu bandingkan dengan prediksi Model A.

## 6. Simpan Model A untuk dashboard

In [ ]:
joblib.dump(final_model, f"{OUT_DIR}/model_a.joblib")
config = {
    "model": BEST, "features": final_cols, "classes": list(final_model.classes_), "display": DISPLAY,
    "window_s": WINDOW_S, "step_s": STEP_S, "target_fps": TARGET_FPS, "frame_width": 640,
    "roi_top": 0.35, "count_line_y": 0.70, "det_conf": 0.2, "imgsz": 640,
    "detectors": {"highway/barat": "yolov8s.pt (COCO), classes [2,3,5,7]",
                  "lalu lintas campuran Indonesia": "output/02_yolo/yolov8s_traffic7.pt, tanpa kelas person"},
    "cv_results": {m: {"accuracy": float(summary.loc[m, ("accuracy", "mean")]), "macro_f1": float(summary.loc[m, ("macro_f1", "mean")])}
                   for m in summary.index},
    "mapping_to_pems_level": {"light": "Low", "medium": "Moderate-High", "heavy": "Severe"},
}
json.dump(config, open(f"{OUT_DIR}/model_a_config.json", "w"), indent=2)
print("Tersimpan di", OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)

## 7. Ringkasan
| Hasil | Lokasi |
|---|---|
| Model A (klasifier) | `output/03_model_a/model_a.joblib` |
| Konfigurasi (fitur, window, detektor, metrik) | `output/03_model_a/model_a_config.json` |
| Fitur 254 klip UCSD | `output/03_model_a/ucsd_features.csv` |
| Timeline + video demo speed layer | `output/03_model_a/timeline_video_uji.csv`, `demo_model_a.mp4` |
| Grafik laporan | `output/03_model_a/figures/` |

**Temuan kunci untuk laporan:**
1. **Domain shift nyata:** detektor fine-tuned (Asia Selatan) gagal di jalan tol Barat; detektor dipilih per domain kamera.
2. Fitur **bebas kalibrasi** (kecepatan dalam panjang-kendaraan/detik, okupansi) cukup untuk mengklasifikasi level kemacetan.
3. Kelas **Macet** paling mudah dikenali; batas **Padat ↔ Lancar** paling sulit.
4. Evaluasi memakai **4 split resmi UCSD** dan macro-F1 — bisa dibandingkan dengan paper Chan & Vasconcelos (2005).

**Sitasi dataset:** A. B. Chan & N. Vasconcelos, *Probabilistic Kernels for the Classification of Auto-Regressive Visual Processes*, CVPR 2005. Video courtesy WSDOT.

---
**Selanjutnya → Notebook 04:** Model B — prediksi kecepatan & level kemacetan **t+1 jam dan t+3 jam** dari master dataset PEMS-BAY (batch layer).